# 5 — Database (ChromaDB)

Creates a persistent ChromaDB vector database from embedded chunks.

| | |
|---|---|
| **Input** | `../data/processed/chunks_embedded.json` |
| **Output** | `../data/db/chroma/` (persistent directory) |
| **Collection** | `taxes_az_qa` |

### Why ChromaDB?
- No server setup — runs as a local file-based DB
- Built-in vector similarity search (HNSW index)
- Stores metadata alongside vectors — no separate SQL DB needed
- `upsert` support — safe to re-run without duplicates

### Safe to re-run
This notebook uses `upsert` not `add`. Running it again after adding new chunks (e.g. long chunks from Sprint 2) will update existing records and insert new ones — no duplicates, no manual cleanup needed.

## 0 — Install

In [1]:
# Run once — comment out after first install
!pip install chromadb tqdm

  Using cached pybase64-1.4.3-cp311-cp311-win_amd64.whl.metadata (9.1 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached kubernetes-35.0.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached durationpy-0.10-py3-none-any.whl.metadata (340 bytes)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached httptools-0.7.1-cp311-cp311-win_amd64.whl.metadata (3.6 

## 1 — Imports & config

In [2]:
import json
import logging
import time
from collections import Counter
from pathlib import Path

import chromadb
from tqdm import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [3]:
# ── Config ────────────────────────────────────────────────────────────────────
INPUT_PATH      = Path("../data/processed/chunks_embedded.json")
DB_PATH         = Path("../data/db/chroma")
COLLECTION_NAME = "taxes_az_qa"
BATCH_SIZE      = 500

print(f"Input : {INPUT_PATH}")
print(f"DB    : {DB_PATH}")
print(f"Collection: {COLLECTION_NAME}")

Input : ..\data\processed\chunks_embedded.json
DB    : ..\data\db\chroma
Collection: taxes_az_qa


## 2 — Load & inspect embedded chunks

In [4]:
with open(INPUT_PATH, encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Total chunks        : {len(chunks):,}")
print(f"Chunk types         : {dict(Counter(c['chunk_type'] for c in chunks))}")
print(f"Embedding model     : {chunks[0]['embedding_model']}")
print(f"Embedding dim       : {chunks[0]['embedding_dim']}")
print(f"Normalized          : {chunks[0]['normalized']}")
print()

# Validate — every chunk must have an embedding
missing = [i for i, c in enumerate(chunks) if not c.get("embedding")]
print(f"Chunks missing embedding: {len(missing)}")
if missing:
    print(f"WARNING: skipping {len(missing)} chunks with no embedding")
    chunks = [c for c in chunks if c.get("embedding")]
    print(f"Proceeding with {len(chunks):,} chunks")

Total chunks        : 3,558
Chunk types         : {'simple': 3558}
Embedding model     : intfloat/multilingual-e5-large
Embedding dim       : 1024
Normalized          : True

Chunks missing embedding: 0


## 3 — Connect to ChromaDB & create collection

In [5]:
DB_PATH.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(DB_PATH))

# get_or_create — safe to re-run
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},  # cosine correct because vectors are normalized
)

print(f"Collection '{COLLECTION_NAME}' ready")
print(f"Existing docs in collection: {collection.count():,}")
print(f"DB path: {DB_PATH.resolve()}")

Collection 'taxes_az_qa' ready
Existing docs in collection: 0
DB path: D:\projects\ai\tax_rag\data\db\chroma


## 4 — Build records and upsert

In [6]:
def build_record(chunk: dict) -> tuple:
    """
    ID format: src{source_id}_c{chunk_index}
    Example  : src10826_c0
    Stable across re-runs — same chunk always gets same ID.
    """
    doc_id    = f"src{chunk['source_id']}_c{chunk['chunk_index']}"
    embedding = chunk["embedding"]
    document  = chunk["chunk_text"]

    metadata = {
        # Identity
        "source_id":       chunk["source_id"],
        "source_url":      chunk["source_url"],
        "chunk_index":     chunk["chunk_index"],
        "total_chunks":    chunk["total_chunks"],
        "chunk_type":      chunk["chunk_type"],
        # Content — stored so chatbot doesn't need to re-read JSON
        "question":        chunk["question"],
        "answer":          chunk["answer"],
        # Quality signals
        "answer_date":     chunk["answer_date"],
        "read_count":      chunk["read_count"],
        "average_rate":    chunk["average_rate"],
        "rate_count":      chunk["rate_count"],
        # Length info
        "question_len":    chunk["question_len"],
        "answer_len":      chunk["answer_len"],
        "combined_len":    chunk["combined_len"],
        # Audit
        "embedding_model": chunk["embedding_model"],
        "embedding_dim":   chunk["embedding_dim"],
        "embedded_at":     chunk["embedded_at"],
    }

    return doc_id, embedding, document, metadata

In [7]:
batches = [chunks[i : i + BATCH_SIZE] for i in range(0, len(chunks), BATCH_SIZE)]
print(f"Upserting {len(chunks):,} chunks in {len(batches):,} batches ...")

total_upserted = 0
t_start = time.time()

for batch_num, batch in enumerate(tqdm(batches, desc="Inserting", unit="batch")):
    ids, embeddings, documents, metadatas = [], [], [], []

    for chunk in batch:
        doc_id, embedding, document, metadata = build_record(chunk)
        ids.append(doc_id)
        embeddings.append(embedding)
        documents.append(document)
        metadatas.append(metadata)

    try:
        collection.upsert(
            ids=ids,
            embeddings=embeddings,
            documents=documents,
            metadatas=metadatas,
        )
        total_upserted += len(batch)
    except Exception as e:
        log.error(f"Batch {batch_num} failed: {e}")

elapsed = time.time() - t_start
print(f"\nDone in {elapsed:.1f}s — upserted {total_upserted:,} chunks")

Upserting 3,558 chunks in 8 batches ...


Inserting: 100%|██████████| 8/8 [00:15<00:00,  1.92s/batch]


Done in 15.3s — upserted 3,558 chunks


## 5 — Verify collection

In [8]:
count = collection.count()
print(f"Total docs in collection: {count:,}")
print()

# Fetch one record and inspect all fields
sample = collection.get(limit=1, include=["documents", "metadatas", "embeddings"])
print("── Sample record ─────────────────────────────────────")
print(f"  id              : {sample['ids'][0]}")
print(f"  document[:100]  : {sample['documents'][0][:100]}")
print(f"  embedding[:5]   : {[round(v,6) for v in sample['embeddings'][0][:5]]}")
print(f"  embedding dim   : {len(sample['embeddings'][0])}")
print()
print("  Metadata fields:")
for k, v in sample['metadatas'][0].items():
    print(f"    {k:<18}: {str(v)[:80]}")

Total docs in collection: 3,558

── Sample record ─────────────────────────────────────
  id              : src10826_c0
  document[:100]  : Sual: Hüquqi şəxsin fəaliyyəti 10 ildir dayandırılmışdır. Təsisçisi Azərbaycan Respublikasının vətən
  embedding[:5]   : [np.float64(0.006775), np.float64(-0.00686), np.float64(-0.01399), np.float64(-0.035928), np.float64(0.023459)]
  embedding dim   : 1024

  Metadata fields:
    question          : Hüquqi şəxsin fəaliyyəti 10 ildir dayandırılmışdır. Təsisçisi Azərbaycan Respubl
    chunk_type        : simple
    answer_len        : 278
    answer            : Bildiririk ki, hüquqi şəxsin ləğv edilməsi proseduru, o cümlədən təqdim olunmalı
    total_chunks      : 1
    read_count        : 62
    embedding_model   : intfloat/multilingual-e5-large
    chunk_index       : 0
    embedded_at       : 2026-05-11T12:18:58.153332+00:00
    question_len      : 374
    combined_len      : 652
    average_rate      : 0
    embedding_dim     : 1024
    source_

## 6 — Test similarity search

In [9]:
# Load the embedding model to test a real query
# (model should still be loaded from 4_embed.ipynb — if not, reload it)
from sentence_transformers import SentenceTransformer

MODEL_NAME    = "intfloat/multilingual-e5-large"
E5_QUERY_PREFIX = "query: "   # IMPORTANT: use 'query: ' prefix at search time

model = SentenceTransformer(MODEL_NAME)
print("Model loaded")

16:36:54 | INFO | No device provided, using cuda:0
16:36:54 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
16:36:54 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large/3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3/modules.json "HTTP/1.1 200 OK"
16:36:54 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
16:36:54 | INFO | Loading SentenceTransformer model from intfloat/multilingual-e5-large.
16:36:55 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
16:36:55 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
16:36:55 | INFO | HTTP Request: HEAD https://hu

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

16:36:57 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
16:36:57 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
16:36:57 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
16:36:57 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
16:36:58 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
16:36:58 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large/3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3/tokenizer_config.json "HTTP/1.1 200 OK"
16:36:58 | INFO | HTTP R

Model loaded


In [10]:
def search(query: str, n_results: int = 5) -> list[dict]:
    """
    Embed a query and retrieve top-n most similar chunks.
    Always prefix with 'query: ' — this is the E5 convention.
    Returns list of result dicts with distance, question, answer.
    """
    query_vec = model.encode(
        [E5_QUERY_PREFIX + query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    results = collection.query(
        query_embeddings=query_vec.tolist(),
        n_results=n_results,
        include=["metadatas", "distances", "documents"],
    )

    output = []
    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        output.append({
            "rank":         i + 1,
            "id":           results["ids"][0][i],
            "distance":     round(results["distances"][0][i], 4),
            "source_id":    meta["source_id"],
            "chunk_type":   meta["chunk_type"],
            "answer_date":  meta["answer_date"],
            "read_count":   meta["read_count"],
            "question":     meta["question"],
            "answer":       meta["answer"],
        })
    return output

In [11]:
# Test with a real Azerbaijani tax question
test_query = "VÖEN-in ləğvi proseduru necədir?"

results = search(test_query, n_results=3)

print(f"Query: {test_query}")
print("=" * 60)
for r in results:
    print(f"Rank {r['rank']} | distance: {r['distance']} | id: {r['id']} | date: {r['answer_date']}")
    print(f"  Q: {r['question'][:120]}")
    print(f"  A: {r['answer'][:120]}")
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: VÖEN-in ləğvi proseduru necədir?
Rank 1 | distance: 0.1278 | id: src9298_c0 | date: 2024-04-26
  Q: Salam. VÖEN-mi necə ləğv edə bilərəm?
  A: Bildiririk ki, vergi uçotundan çıxmaq üçün vergi ödəyicisi və ya onun səlahiyyətli nümayəndəsi (notarial qaydada təsdiq 

Rank 2 | distance: 0.1316 | id: src1516_c0 | date: 2020-04-22
  Q: VÖEN-in ləğv edilməsi.
  A: Bildiririk ki, vergi uçotunuzun ləğv edilməsi üçün “Fiziki şəxsin vergi uçotundan çıxarılması haqqında ərizə” və uçota a

Rank 3 | distance: 0.132 | id: src4651_c0 | date: 2021-05-17
  Q: 2004237452 nömrəli VÖEN-in ləğv olunması ücün uzun muddətdir ki, muraciət etmişəm. Ancaq hələ ləğv olunmayıb, xahiş olun
  A: Bildiririk ki, fiziki şəxs vergi uçotunun ləğv edilməsi haqda müraciət etdikdə, vergi orqanında müvafiq prosedurlar (akt



In [12]:
# Try another query
test_query2 = "Gəlir vergisi hansı faizlə hesablanır?"

results2 = search(test_query2, n_results=3)

print(f"Query: {test_query2}")
print("=" * 60)
for r in results2:
    print(f"Rank {r['rank']} | distance: {r['distance']} | id: {r['id']}")
    print(f"  Q: {r['question'][:120]}")
    print(f"  A: {r['answer'][:120]}")
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: Gəlir vergisi hansı faizlə hesablanır?
Rank 1 | distance: 0.1303 | id: src3942_c0
  Q: Salam. P2P lendinq platformalarında qazanılan faizə görə hansı vergi ödənilməlidir? Mən əgər ki, qazanılan faizi platfor
  A: Bildiririk ki, Vergi Məcəlləsinin 99.3.8-ci maddəsinə əsasən vergi ödəyicisinin aktivlərinin ilkin qiymətinin artdığını 

Rank 2 | distance: 0.1349 | id: src4629_c0
  Q: Salam. Kriptovalyuta vergisi haqqında məlumat almaq istəyirdim. Neçə faiz vergi tutulur?
  A: Bildiririk ki, Vergi Məcəlləsinin 99.3.8-ci maddəsinə əsasən vergi ödəyicisinin aktivlərinin (o cümlədən kriptovalyutanı

Rank 3 | distance: 0.1353 | id: src3337_c0
  Q: Salam. Mən Azərbaycanda Qeyri-Hökümət Təşkilatlarının kommersiya yolu ilə əldə etdikləri gəlirin hansi faizlə hesablandı
  A: Bildiririk ki, Azərbaycan Respublikasında rezident və qeyri-rezident müəssisələr, habelə sahibkarlıq fəaliyyətindən gəli



## 7 — Summary

In [14]:
print("═" * 54)
print("  DATABASE READY")
print("═" * 54)
print(f"  Collection    : {COLLECTION_NAME}")
print(f"  Total docs    : {collection.count():,}")
print(f"  DB path       : {DB_PATH.resolve()}")
print(f"  Distance space: cosine")
print(f"  Time elapsed  : {elapsed:.1f}s")
print("═" * 54)

══════════════════════════════════════════════════════
  DATABASE READY
══════════════════════════════════════════════════════
  Collection    : taxes_az_qa
  Total docs    : 3,558
  DB path       : D:\projects\ai\tax_rag\data\db\chroma
  Distance space: cosine
  Time elapsed  : 15.3s
══════════════════════════════════════════════════════
